In [49]:
from app.db_utils import (
    reset_db,
    drop_table, 
    to_df, 
    get_table_names, 
    count_records, 
    backup_table,
    get_table_schema,
    get_session,
    execute_raw_query
)
import asyncio
# from app.tasks import trigger_contest_sync
from app.crud import update_upcoming_contests
from app.database import SessionLocal
from app.models import *
from app.codeforces_api import cf_api
import random
from datetime import datetime
import requests

DEFAULT_PASS = "devpass"
SPECIAL_USERS = ["negative-xp", "roomTemperatureIQ"]

In [51]:
df = to_df("group_memberships")
# df[df.group_id == 'private']

df

,user_id,group_id,role,user_group_rating,user_group_max_rating,cf_handle,timestamp
0,Assem_albitar,main,user,1474,1474,Assem_albitar,2025-06-01 23:03:09.961080
1,FatemehA,main,user,1672,1672,FatemehA,2025-06-01 23:03:09.961080
2,Aristides,main,user,0,0,Aristides,2025-06-01 23:03:09.961080
3,Denislav_Manev,main,user,0,0,Denislav_Manev,2025-06-01 23:03:09.961080
4,Under_,main,user,0,0,Under_,2025-06-01 23:03:09.961080
...,...,...,...,...,...,...,...
6147,fr200110217102,main,user,1965,2377,fr200110217102,2025-06-01 23:03:09.961080
6148,hardenisthegoat,main,user,1829,1919,hardenisthegoat,2025-06-01 23:03:09.961080
6149,ishmeal,main,user,2231,2231,ishmeal,2025-06-01 23:03:09.961080
6150,ji_114514,main,user,2073,2083,ji_114514,2025-06-01 23:03:09.961080


In [33]:
print("generating some non-member users for seeding requests")
non_member_users = []
already_added = set()
users = db.query(User).all()
for user in users:
    already_added.add(user.cf_handle)
handle_set = cf_api.user_ratedList()
while len(non_member_users) < 200:
    idx = random.randint(0, len(handle_set))
    handle = handle_set[idx]["handle"]
    if handle not in already_added:
        non_member_users.append(
            User(
                user_id=handle,
                role=Role.user,
                cf_handle=handle,
                email_id=f"{handle}@example.com",
                hashed_password=hash_password(DEFAULT_PASS),
            )
        )
        already_added.add(handle)

db.add_all(non_member_users)
db.commit()

# print("non-member users generated in", f"{time.perf_counter() - t0:.1f}s")

# banner("generating request objects")
reqs = []
new_memberships = []
for user in non_member_users:

    req_type = int(random.random() *3 )

    if req_type == 0:
        # unresolved request
        reqs.append(
            Request(
                request_id=f"req{len(reqs) + 1}",
                group_id="main",
                user_id=user.user_id,
            )
        )
    elif req_type == 1:
        # accepted request
        resolver = random.choice(SPECIAL_USERS)
        reqs.append(
            Request(
                request_id=f"req{len(reqs) + 1}",
                group_id="main",
                user_id=user.user_id,
                accepted=True,
                resolved=True,
                resolver_user_id=resolver,
                resolver_cf_handle=resolver,
                resolve_timestamp=datetime.now(),
            )
        )
        new_memberships.append(
            GroupMembership(
                user_id=user.user_id,
                group_id="main",
                role=Role.user,
            )
        )
    else:
        # rejected request
        resolver = random.choice(SPECIAL_USERS)
        reqs.append(
            Request(
                request_id=f"req{len(reqs) + 1}",
                group_id="main",
                user_id=user.user_id,
                accepted=False,
                resolved=True,
                resolver_user_id=resolver,
                resolver_cf_handle=resolver,
                resolve_timestamp=datetime.now(),
            )
        )

db.add_all(reqs)
db.add_all(new_memberships)
db.commit()


generating some non-member users for seeding requests


In [36]:
to_df("requests")

,request_id,group_id,user_id,resolved,accepted,resolve_timestamp,resolver_user_id,resolver_cf_handle,timestamp
0,req1,main,rockstarlife999,True,False,2025-06-02 03:40:18.206694,roomTemperatureIQ,roomTemperatureIQ,2025-06-01 22:10:18.205143
1,req2,main,brokie,True,False,2025-06-02 03:40:18.207048,roomTemperatureIQ,roomTemperatureIQ,2025-06-01 22:10:18.205143
2,req3,main,sanjana_8477,True,True,2025-06-02 03:40:18.207346,roomTemperatureIQ,roomTemperatureIQ,2025-06-01 22:10:18.205143
3,req4,main,leafeartheater813,True,False,2025-06-02 03:40:18.207653,roomTemperatureIQ,roomTemperatureIQ,2025-06-01 22:10:18.205143
4,req5,main,syfy24,False,None,NaT,None,None,2025-06-01 22:10:18.205143
...,...,...,...,...,...,...,...,...,...
195,req196,main,adfjffff,True,False,2025-06-02 03:40:18.247874,roomTemperatureIQ,roomTemperatureIQ,2025-06-01 22:10:18.205143
196,req197,main,trankhaihung,False,None,NaT,None,None,2025-06-01 22:10:18.205143
197,req198,main,alayadcruz09,False,None,NaT,None,None,2025-06-01 22:10:18.205143
198,req199,main,ExpertCoder0069,True,True,2025-06-02 03:40:18.248384,roomTemperatureIQ,roomTemperatureIQ,2025-06-01 22:10:18.205143


In [23]:
to_df("contest_participations")

,user_id,group_id,contest_id,rank,delta,rating_before,rating_after,timestamp
0,testUser1726,main,c0,None,None,1500,None,2025-05-25 16:44:27.268643
1,testUser3835,main,c0,None,None,1500,None,2025-05-25 16:44:27.268643
2,testUser3036,main,c0,None,None,1500,None,2025-05-25 16:44:27.268643
3,testUser5213,main,c0,None,None,1500,None,2025-05-25 16:44:27.268643
4,testUser4940,main,c0,None,None,1500,None,2025-05-25 16:44:27.268643
...,...,...,...,...,...,...,...,...
47167,testUser3160,g29,c3,None,None,1500,None,2025-05-25 16:44:27.268643
47168,testUser4067,g29,c3,None,None,1500,None,2025-05-25 16:44:27.268643
47169,testUser2897,g29,c3,None,None,1500,None,2025-05-25 16:44:27.268643
47170,testUser1125,g29,c3,None,None,1500,None,2025-05-25 16:44:27.268643
